# FS-UNeXt: Fully supervised UNeXt for blood cell segmentation

**UNeXt**: Lightweight CNN (~1.5M parameters) – the model deployed by DACL.

This notebook is **self-contained** (no repository cloning needed):
1. Experiment configuration
2. Data preparation & splitting
3. **Data augmentation** – configuration + visual preview
4. Dataset / DataLoader
5. UNeXt model
6. Loss & evaluation metrics
7. Training loop
8. Experiments at each label ratio
9. Results: table, training curves, prediction overlays

**Before running on Kaggle**
- *Settings → Accelerator*: **GPU T4 x2** or **GPU P100**
- *Settings → Internet*: **On** (optional)
- *Add Input*: the Kaggle Dataset holding the Blood Cell Segmentation data (1328 images + 1328 masks)
- Run everything: *Run All*, or *Save Version → Save & Run All (Commit)* to run in the background for up to 12 h

## 1. Experiment configuration

In [ ]:
# ================================================================
#                    EXPERIMENT CONFIGURATION
#  Edit this cell only. All cells below read their settings from CFG.
# ================================================================
class CFG:
    # ---------------- PATHS ----------------
    # RAW dataset folder on Kaggle ("Input" panel on the right -> copy path button).
    # Files are scanned recursively and *_image / *_mask are paired automatically,
    # so the sub-folder layout does not matter.
    RAW_DIR = "/kaggle/input/blood-cell-segmentation"
    # Where prepared data and results are written. On Kaggle only /kaggle/working is kept.
    WORK_DIR = "/kaggle/working"

    # ---------------- MODEL ----------------
    MODEL = "unext"
    # UNeXt channel configuration:
    #   "dacl"     : (32, 64, 128, 160, 256) - as stated in the DACL paper -> 1.58M params
    #   "original" : (16, 32, 128, 160, 256) - as in the original UNeXt    -> 1.47M params
    # (the DACL paper reports 1.47M but states the "dacl" channels -> inconsistent).
    # Use the SAME configuration for FS-UNeXt and the UNeXt inside DACL.
    UNEXT_VARIANT = "dacl"
    NUM_CLASSES = 2          # 0 = background, 1 = cell (binary masks)

    # ---------------- DATA ----------------
    # The 1600x1200 originals are resized ONCE to this size (keeps the 4:3 ratio), so the
    # DataLoader never decodes large images during training -> much faster.
    PREP_SIZE = (640, 480)   # (width, height)
    # Masks are stored as JPG (compression noise) -> pixels > threshold are cell.
    MASK_THRESHOLD = 127
    # Network input size. 224 matches Swin-Unet (window 7, ImageNet weights at 224).
    # KEEP 224 for both UNeXt and Swin-Unet for a fair comparison.
    IMG_SIZE = 224
    # Train/val/test ratios and split seed. KEEP the same seed across notebooks so every
    # model (FS, DACL, baselines) uses the SAME data split.
    SPLIT_RATIOS = (0.7, 0.1, 0.2)
    SPLIT_SEED = 2026
    # Label ratios (%) to run. 100 = the whole training set (upper bound).
    # 5/10/20 = only the labeled part of the SSL experiments (SSL lower bound).
    LABELED_RATIOS = [5, 10, 20, 100]

    # ---------------- TRAINING ----------------
    # Train for a fixed NUMBER OF ITERATIONS (not epochs): same number of updates at every ratio.
    MAX_ITERS = 30000
    # Batch size. UNeXt is very light: 8 fits a 16GB GPU easily (16 also works).
    BATCH_SIZE = 8
    # SGD as in the DACL paper; poly learning-rate decay: lr*(1 - it/max_it)^0.9.
    BASE_LR = 0.01
    MOMENTUM = 0.9
    WEIGHT_DECAY = 1e-4
    # FP16 mixed precision: ~1.5-2x faster and less GPU memory.
    USE_AMP = True
    # Data-loading workers (Kaggle has 4 CPUs).
    NUM_WORKERS = 4
    # Seeds. Add 2027, 2028 to repeat runs and report mean ± std.
    SEEDS = [2026]

    # ---------------- EVALUATION & LOGGING ----------------
    # Validate every VAL_INTERVAL iterations; save best.pth when the Dice improves.
    VAL_INTERVAL = 500
    LOG_INTERVAL = 100

    # ---------------- DRY RUN ----------------
    # True -> each experiment runs only 300 iterations, to test the whole notebook in minutes.
    # REMEMBER to set it back to False for the real run.
    DEBUG = False


if CFG.DEBUG:
    CFG.MAX_ITERS, CFG.VAL_INTERVAL, CFG.LOG_INTERVAL = 300, 100, 50

In [ ]:
import csv, json, os, random, re, shutil, time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
from scipy import ndimage
from skimage.color import hed2rgb, rgb2hed
from timm.layers import DropPath, to_2tuple, trunc_normal_
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path(CFG.WORK_DIR) / "data" / "processed"
SPLIT_DIR = Path(CFG.WORK_DIR) / "data" / "splits"
RUN_ROOT = Path(CFG.WORK_DIR) / "runs"
print("torch", torch.__version__, "| albumentations", A.__version__, "| device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Data preparation & splitting

- Pair images and masks by name (`123_image.png` ↔ `123_mask.jpg`), resize, binarize masks.
- **70/10/20** split (train/val/test) with a fixed seed; the first **2/5/10/20%** of train are the labeled subsets.
- This cell **skips itself** if the data was already prepared in the current session.

> Check the log: **1328 pairs** must be found, and the cell-pixel ratio must not be close to 0 or 1.

In [ ]:
EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
MASK_WORDS = ("mask", "gt", "label", "ground")
SUFFIX_RE = re.compile(r"[ _\-]?(mask|masks|gt|label|labels|image|images|img|original)$", re.I)


def _is_mask(p):
    return any(w in p.stem.lower() for w in MASK_WORDS) or any(w in p.parent.name.lower() for w in MASK_WORDS)


def prepare_dataset():
    if (DATA_DIR / "DONE").exists():
        print("Data already prepared - skipping.")
        return
    files = [p for p in Path(CFG.RAW_DIR).rglob("*") if p.suffix.lower() in EXTS]
    images, masks = {}, {}
    for p in files:
        (masks if _is_mask(p) else images)[SUFFIX_RE.sub("", p.stem).strip().lower()] = p
    keys = sorted(set(images) & set(masks))
    print(f"Found {len(images)} images, {len(masks)} masks -> {len(keys)} pairs")
    if not keys:
        raise RuntimeError("No pairs found - check CFG.RAW_DIR and the file naming.")
    (DATA_DIR / "images").mkdir(parents=True, exist_ok=True)
    (DATA_DIR / "masks").mkdir(parents=True, exist_ok=True)
    fg = []
    for i, k in enumerate(tqdm(keys, desc="Preparing")):
        img = cv2.imread(str(images[k]), cv2.IMREAD_COLOR)
        msk = cv2.imread(str(masks[k]), cv2.IMREAD_GRAYSCALE)
        if img is None or msk is None:
            print("  skipping unreadable file:", k)
            continue
        img = cv2.resize(img, CFG.PREP_SIZE, interpolation=cv2.INTER_AREA)
        msk = cv2.resize(msk, CFG.PREP_SIZE, interpolation=cv2.INTER_NEAREST)
        msk = np.where(msk > CFG.MASK_THRESHOLD, 255, 0).astype(np.uint8)
        cv2.imwrite(str(DATA_DIR / "images" / f"{i:04d}.png"), img)
        cv2.imwrite(str(DATA_DIR / "masks" / f"{i:04d}.png"), msk)
        fg.append((msk > 0).mean())
    (DATA_DIR / "DONE").write_text("ok")
    print(f"Saved {len(fg)} pairs | mean cell-pixel ratio: {np.mean(fg):.3f}")


def split_dataset():
    names = sorted(p.stem for p in (DATA_DIR / "images").glob("*.png"))
    random.Random(CFG.SPLIT_SEED).shuffle(names)
    n = len(names)
    n_tr, n_va = round(CFG.SPLIT_RATIOS[0] * n), round(CFG.SPLIT_RATIOS[1] * n)
    split = {"train": names[:n_tr], "val": names[n_tr:n_tr + n_va], "test": names[n_tr + n_va:]}
    for pct in (2, 5, 10, 20):  # 2% for the extreme low-label experiment
        split[f"labeled_{pct}"] = split["train"][:int(len(split["train"]) * pct / 100)]
    SPLIT_DIR.mkdir(parents=True, exist_ok=True)
    for k, v in split.items():
        (SPLIT_DIR / f"{k}.txt").write_text("\n".join(v) + "\n", encoding="utf-8")
    print({k: len(v) for k, v in split.items()})
    return split


prepare_dataset()
SPLIT = split_dataset()

## 3. Data augmentation

Applied to **labeled training images only**; validation/test are only resized + normalized.
Augmentation is **online**: every time an image is loaded a new random variant is generated, so over
30k iterations the model practically never sees the same image twice – no need to write extra
augmented images to disk (offline augmentation).

| Group | Transform | What it simulates |
|---|---|---|
| Geometric | Horizontal/vertical flip, 90° rotation | Cells have no canonical orientation |
| | Affine (scale, small rotation, translation) | Different magnification and slide placement |
| | Elastic | Slight cell-shape deformation |
| Color / stain | Color jitter | Different lamps, cameras, white balance |
| | **HED stain jitter** | Stronger/weaker staining between slides and labs |
| | Gamma, CLAHE | Different contrast and exposure |
| Image quality | Gaussian blur, Gaussian noise | Imperfect focus, sensor noise |

**`LEVEL`** – use it for an augmentation ablation:
- `"none"`: no augmentation · `"basic"`: geometric only · `"full"`: geometric + color/stain + image quality

In [ ]:
# ================================================================
#                DATA AUGMENTATION CONFIGURATION
#  p = probability of applying each transform to an image (0 = off, 1 = always)
# ================================================================
class AUG:
    LEVEL = "full"            # "none" | "basic" | "full"

    # ---- Geometric (applied to the image AND the mask) ----
    HFLIP_P = 0.5             # horizontal flip
    VFLIP_P = 0.5             # vertical flip
    ROT90_P = 0.5             # rotate by 90/180/270 degrees
    AFFINE_P = 0.5            # affine transform:
    SCALE = (0.85, 1.15)      #   scale 0.85-1.15x (magnification differences)
    ROTATE = (-15, 15)        #   extra rotation ±15 degrees
    TRANSLATE = (-0.05, 0.05) #   shift ±5% of the image size
    ELASTIC_P = 0.2           # elastic deformation
    ELASTIC_ALPHA = 30        #   strength (too large distorts cell shapes unrealistically)
    ELASTIC_SIGMA = 6         #   smoothness of the displacement field

    # ---- Color / stain (image ONLY, the mask is unchanged) ----
    COLOR_JITTER_P = 0.5
    BRIGHTNESS = 0.2          # brightness ±20%
    CONTRAST = 0.2            # contrast ±20%
    SATURATION = 0.2          # saturation ±20%
    HUE = 0.05                # hue ±0.05 (kept small so cell colors stay plausible)
    HED_P = 0.5               # stain jitter in HED color space (Hematoxylin-Eosin-DAB):
    HED_SIGMA = 0.05          #   per-channel multiplicative factor ~ U(1-σ, 1+σ)
    HED_BIAS = 0.005          #   per-channel additive shift ~ U(-b, b). skimage HED values lie in
                              #   ~[0, 0.1], so b must be SMALL; b = 0.02 already whitened the background.
                              #   (Blood smears use Giemsa/Wright stain; HED is a reasonable
                              #    approximation of the purple/pink dyes and standard in histopathology.)
    GAMMA_P = 0.2
    GAMMA = (80, 120)         # gamma 0.8-1.2 (in %)
    CLAHE_P = 0.1             # contrast-limited adaptive histogram equalization

    # ---- Image quality ----
    BLUR_P = 0.1              # Gaussian blur (defocus)
    NOISE_P = 0.1             # Gaussian noise (sensor noise)
    NOISE_STD = (0.01, 0.04)  #   noise std on a [0, 1] scale (0.04 ≈ ±10 grey levels)

    # ---- Normalization ----
    # ImageNet statistics: required by the pretrained Swin-Unet; shared with UNeXt so
    # both models receive identical inputs.
    MEAN = (0.485, 0.456, 0.406)
    STD = (0.229, 0.224, 0.225)

In [ ]:
# albumentations 1.x and 2.x use different argument names -> two helpers support both.
ALB_V2 = int(A.__version__.split(".")[0]) >= 2


def _affine():
    # Empty areas created by rotation/scaling are filled by reflection (no artificial black borders).
    border = {"border_mode": cv2.BORDER_REFLECT_101} if ALB_V2 else {"mode": cv2.BORDER_REFLECT_101}
    return A.Affine(scale=AUG.SCALE, rotate=AUG.ROTATE, translate_percent=AUG.TRANSLATE,
                    p=AUG.AFFINE_P, **border)


def _noise():
    if ALB_V2:
        return A.GaussNoise(std_range=AUG.NOISE_STD, p=AUG.NOISE_P)
    var = tuple((s * 255) ** 2 for s in AUG.NOISE_STD)  # 1.x uses variance on a 0-255 scale
    return A.GaussNoise(var_limit=var, p=AUG.NOISE_P)


def hed_jitter(img, **kwargs):
    """HED stain jitter (Tellez et al., 2018): RGB -> HED, perturb each channel, back to RGB."""
    hed = rgb2hed(img.astype(np.float32) / 255.0)
    alpha = np.random.uniform(1 - AUG.HED_SIGMA, 1 + AUG.HED_SIGMA, 3)
    beta = np.random.uniform(-AUG.HED_BIAS, AUG.HED_BIAS, 3)
    rgb = hed2rgb(hed * alpha + beta)
    return (np.clip(rgb, 0, 1) * 255).astype(np.uint8)


def build_train_transform(level=None, normalize=True):
    level = level or AUG.LEVEL
    t = [A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE)]
    if level in ("basic", "full"):
        t += [
            A.HorizontalFlip(p=AUG.HFLIP_P),
            A.VerticalFlip(p=AUG.VFLIP_P),
            A.RandomRotate90(p=AUG.ROT90_P),
            _affine(),
            A.ElasticTransform(alpha=AUG.ELASTIC_ALPHA, sigma=AUG.ELASTIC_SIGMA, p=AUG.ELASTIC_P),
        ]
    if level == "full":
        t += [
            A.ColorJitter(brightness=AUG.BRIGHTNESS, contrast=AUG.CONTRAST,
                          saturation=AUG.SATURATION, hue=AUG.HUE, p=AUG.COLOR_JITTER_P),
            A.Lambda(image=hed_jitter, p=AUG.HED_P),
            A.RandomGamma(gamma_limit=AUG.GAMMA, p=AUG.GAMMA_P),
            A.CLAHE(p=AUG.CLAHE_P),
            A.GaussianBlur(blur_limit=(3, 5), p=AUG.BLUR_P),
            _noise(),
        ]
    if normalize:
        t += [A.Normalize(mean=AUG.MEAN, std=AUG.STD), ToTensorV2()]
    return A.Compose(t)


def build_eval_transform():
    return A.Compose([A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
                      A.Normalize(mean=AUG.MEAN, std=AUG.STD), ToTensorV2()])


def show_augmentations(name=None, n=7, level=None):
    """Show the original + n augmented variants; green contour = transformed mask."""
    name = name or SPLIT["train"][0]
    img = cv2.cvtColor(cv2.imread(str(DATA_DIR / "images" / f"{name}.png")), cv2.COLOR_BGR2RGB)
    msk = (cv2.imread(str(DATA_DIR / "masks" / f"{name}.png"), 0) > 127).astype(np.uint8)
    tf = build_train_transform(level, normalize=False)
    base = A.Compose([A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE)])(image=img, mask=msk)
    samples = [("Original", base["image"], base["mask"])]
    for i in range(n):
        o = tf(image=img, mask=msk)
        samples.append((f"Variant {i + 1}", o["image"], o["mask"]))
    fig, axes = plt.subplots(2, (n + 2) // 2, figsize=(3 * ((n + 2) // 2), 6))
    for ax, (title, im, m) in zip(axes.ravel(), samples):
        vis = im.copy()
        cnts, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(vis, cnts, -1, (0, 255, 0), 1)
        ax.imshow(vis); ax.set_title(title); ax.axis("off")
    plt.suptitle(f"Data augmentation – LEVEL = {level or AUG.LEVEL}")
    plt.tight_layout(); plt.show()


show_augmentations()

Compare the levels (re-run the cell to see other random variants):

In [ ]:
show_augmentations(level="basic", n=3)

## 4. Dataset / DataLoader

In [ ]:
class BloodCellDataset(Dataset):
    def __init__(self, names, transform):
        self.names, self.transform = list(names), transform

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        name = self.names[i]
        img = cv2.cvtColor(cv2.imread(str(DATA_DIR / "images" / f"{name}.png")), cv2.COLOR_BGR2RGB)
        msk = (cv2.imread(str(DATA_DIR / "masks" / f"{name}.png"), 0) > 127).astype(np.uint8)
        o = self.transform(image=img, mask=msk)
        return {"image": o["image"], "mask": torch.as_tensor(o["mask"]).long(), "name": name}


def infinite_loader(loader):
    """Iterate a DataLoader forever (iteration-based training)."""
    while True:
        for b in loader:
            yield b

## 5. UNeXt model

CNN with tokenized / shifted MLPs (Valanarasu & Patel, MICCAI 2022).

In [ ]:
"""UNeXt: MLP-based Rapid Medical Image Segmentation Network (Valanarasu & Patel, MICCAI 2022).

Re-implemented from the official repo (jeya-maria-jose/UNeXt-pytorch) with configurable channels.
DACL states encoder channels {32, 64, 128, 160, 256} (see UNEXT_DIMS in __init__.py).

Architecture:
  - 3 convolutional stages (Conv -> BN -> MaxPool -> ReLU), each halving the resolution
  - 2 "Tokenized MLP" (shifted MLP) stages at depth + bottleneck
  - Symmetric decoder; skip connections use ADDITION instead of concatenation
"""


class DWConv(nn.Module):
    """3x3 depth-wise convolution over the token sequence (implicit positional encoding)."""

    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, 3, 1, 1, groups=dim)

    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.transpose(1, 2).reshape(B, C, H, W)
        x = self.dwconv(x)
        return x.flatten(2).transpose(1, 2)


class ShiftMLP(nn.Module):
    """Shifted MLP: shift channel groups along height, then width, before each FC layer,
    so the MLP "sees" a local neighbourhood like a convolution, but cheaper."""

    def __init__(self, in_features, hidden_features=None, out_features=None, drop=0.0, shift_size=5):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.dwconv = DWConv(hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)
        self.shift_size = shift_size
        self.pad = shift_size // 2

    def _shift(self, x, H, W, dim):
        """Split channels into shift_size groups; group k is rolled by k-pad pixels along axis dim."""
        B, N, C = x.shape
        xn = x.transpose(1, 2).reshape(B, C, H, W)
        xn = F.pad(xn, (self.pad, self.pad, self.pad, self.pad), "constant", 0)
        chunks = torch.chunk(xn, self.shift_size, 1)
        shifted = [torch.roll(c, s, dim) for c, s in zip(chunks, range(-self.pad, self.pad + 1))]
        xs = torch.cat(shifted, 1)
        xs = torch.narrow(xs, 2, self.pad, H)
        xs = torch.narrow(xs, 3, self.pad, W)
        return xs.reshape(B, C, H * W).transpose(1, 2)

    def forward(self, x, H, W):
        x = self._shift(x, H, W, dim=2)  # shift along height
        x = self.fc1(x)
        x = self.dwconv(x, H, W)
        x = self.act(x)
        x = self.drop(x)
        x = self._shift(x, H, W, dim=3)  # shift along width
        x = self.fc2(x)
        return self.drop(x)


class ShiftedBlock(nn.Module):
    def __init__(self, dim, mlp_ratio=1.0, drop=0.0, drop_path=0.0):
        super().__init__()
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = ShiftMLP(dim, int(dim * mlp_ratio), drop=drop)
        self.drop_path = DropPath(drop_path) if drop_path > 0 else nn.Identity()

    def forward(self, x, H, W):
        return x + self.drop_path(self.mlp(self.norm2(x), H, W))


class OverlapPatchEmbed(nn.Module):
    """3x3 stride-2 conv -> halves the resolution and turns the feature map into tokens."""

    def __init__(self, in_chans, embed_dim, patch_size=3, stride=2):
        super().__init__()
        self.proj = nn.Conv2d(in_chans, embed_dim, patch_size, stride, patch_size // 2)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        _, _, H, W = x.shape
        x = self.norm(x.flatten(2).transpose(1, 2))
        return x, H, W


class UNext(nn.Module):
    def __init__(self, num_classes=2, in_chans=3, dims=(32, 64, 128, 160, 256), drop_path_rate=0.0):
        """
        num_classes: number of output classes (2 = background + cell).
        dims: channels of the 5 encoder stages. The first 3 are the conv stages,
              the last 2 the tokenized-MLP stage and the bottleneck.
              Original UNeXt: (16, 32, 128, 160, 256); DACL: (32, 64, 128, 160, 256).
        """
        super().__init__()
        c1, c2, c3, c4, c5 = dims
        # ---- Encoder: convolutional stages ----
        self.encoder1 = nn.Conv2d(in_chans, c1, 3, 1, 1)
        self.encoder2 = nn.Conv2d(c1, c2, 3, 1, 1)
        self.encoder3 = nn.Conv2d(c2, c3, 3, 1, 1)
        self.ebn1, self.ebn2, self.ebn3 = nn.BatchNorm2d(c1), nn.BatchNorm2d(c2), nn.BatchNorm2d(c3)
        # ---- Encoder: tokenized MLP ----
        self.patch_embed3 = OverlapPatchEmbed(c3, c4)
        self.patch_embed4 = OverlapPatchEmbed(c4, c5)
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, 3)]
        self.block1 = nn.ModuleList([ShiftedBlock(c4, drop_path=dpr[0])])
        self.block2 = nn.ModuleList([ShiftedBlock(c5, drop_path=dpr[1])])
        self.norm3, self.norm4 = nn.LayerNorm(c4), nn.LayerNorm(c5)
        # ---- Decoder ----
        self.dblock1 = nn.ModuleList([ShiftedBlock(c4, drop_path=dpr[0])])
        self.dblock2 = nn.ModuleList([ShiftedBlock(c3, drop_path=dpr[1])])
        self.dnorm3, self.dnorm4 = nn.LayerNorm(c4), nn.LayerNorm(c3)
        self.decoder1 = nn.Conv2d(c5, c4, 3, 1, 1)
        self.decoder2 = nn.Conv2d(c4, c3, 3, 1, 1)
        self.decoder3 = nn.Conv2d(c3, c2, 3, 1, 1)
        self.decoder4 = nn.Conv2d(c2, c1, 3, 1, 1)
        self.decoder5 = nn.Conv2d(c1, c1, 3, 1, 1)
        self.dbn1, self.dbn2 = nn.BatchNorm2d(c4), nn.BatchNorm2d(c3)
        self.dbn3, self.dbn4 = nn.BatchNorm2d(c2), nn.BatchNorm2d(c1)
        self.final = nn.Conv2d(c1, num_classes, 1)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    @staticmethod
    def _to_map(x, B, H, W):
        return x.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()

    @staticmethod
    def _up(x):
        return F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)

    def forward(self, x):
        B = x.shape[0]
        # Encoder - conv stages (1/2, 1/4, 1/8)
        out = F.relu(F.max_pool2d(self.ebn1(self.encoder1(x)), 2, 2)); t1 = out
        out = F.relu(F.max_pool2d(self.ebn2(self.encoder2(out)), 2, 2)); t2 = out
        out = F.relu(F.max_pool2d(self.ebn3(self.encoder3(out)), 2, 2)); t3 = out
        # Tokenized MLP (1/16)
        out, H, W = self.patch_embed3(out)
        for blk in self.block1:
            out = blk(out, H, W)
        out = self._to_map(self.norm3(out), B, H, W); t4 = out
        # Bottleneck (1/32)
        out, H, W = self.patch_embed4(out)
        for blk in self.block2:
            out = blk(out, H, W)
        out = self._to_map(self.norm4(out), B, H, W)

        # Decoder
        out = F.relu(self._up(self.dbn1(self.decoder1(out)))) + t4
        _, _, H, W = out.shape
        out = out.flatten(2).transpose(1, 2)
        for blk in self.dblock1:
            out = blk(out, H, W)
        out = self._to_map(self.dnorm3(out), B, H, W)

        out = F.relu(self._up(self.dbn2(self.decoder2(out)))) + t3
        _, _, H, W = out.shape
        out = out.flatten(2).transpose(1, 2)
        for blk in self.dblock2:
            out = blk(out, H, W)
        out = self._to_map(self.dnorm4(out), B, H, W)

        out = F.relu(self._up(self.dbn3(self.decoder3(out)))) + t2
        out = F.relu(self._up(self.dbn4(self.decoder4(out)))) + t1
        out = F.relu(self._up(self.decoder5(out)))
        return self.final(out)  # logits (B, num_classes, H, W)

In [ ]:
def build_model(pretrained=True):
    """pretrained is ignored: UNeXt is always trained from scratch (as in the paper)."""
    dims = {"dacl": (32, 64, 128, 160, 256), "original": (16, 32, 128, 160, 256)}[CFG.UNEXT_VARIANT]
    return UNext(num_classes=CFG.NUM_CLASSES, dims=dims)


_m = build_model()
print(f"UNeXt ({CFG.UNEXT_VARIANT}): {sum(p.numel() for p in _m.parameters()) / 1e6:.2f}M params")
print("Output:", tuple(_m(torch.randn(1, 3, CFG.IMG_SIZE, CFG.IMG_SIZE)).shape))
del _m

## 6. Loss & evaluation metrics

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.n = n_classes

    def forward(self, logits, target):
        prob = torch.softmax(logits, 1)
        oh = F.one_hot(target, self.n).permute(0, 3, 1, 2).float()
        inter = (prob * oh).sum((0, 2, 3))
        union = prob.sum((0, 2, 3)) + oh.sum((0, 2, 3))
        return 1 - ((2 * inter + 1e-5) / (union + 1e-5)).mean()


DICE = DiceLoss(CFG.NUM_CLASSES)


def sup_loss(logits, target):
    """L_sup = Cross-Entropy + Dice (as in DACL)."""
    return F.cross_entropy(logits, target) + DICE(logits, target)


def _surface(m):
    return m ^ ndimage.binary_erosion(m, structure=ndimage.generate_binary_structure(2, 1))


def binary_metrics(pred, gt):
    """DSC, Jaccard, HD95, ASD for binary masks. HD95/ASD in PIXELS at IMG_SIZE resolution."""
    pred, gt = pred.astype(bool), gt.astype(bool)
    ps, gs = pred.sum(), gt.sum()
    if ps == 0 and gs == 0:
        return {"dsc": 1.0, "jaccard": 1.0, "hd95": 0.0, "asd": 0.0}
    inter = (pred & gt).sum()
    dsc, jac = 2 * inter / (ps + gs), inter / (pred | gt).sum()
    if ps == 0 or gs == 0:  # one side empty -> maximum distance penalty (image diagonal)
        d = float(np.hypot(*pred.shape))
        return {"dsc": float(dsc), "jaccard": float(jac), "hd95": d, "asd": d}
    sp, sg = _surface(pred), _surface(gt)
    d = np.concatenate([ndimage.distance_transform_edt(~sg)[sp], ndimage.distance_transform_edt(~sp)[sg]])
    return {"dsc": float(dsc), "jaccard": float(jac), "hd95": float(np.percentile(d, 95)), "asd": float(d.mean())}

## 7. Training loop

- Each experiment is saved to `runs/<model>_<ratio>p_s<seed>/`: `best.pth`, `last.pth`, `log.csv`, `config.json`, `test_per_image.csv`, `test_summary.json`.
- **Resumes automatically** from `last.pth` if an experiment is unfinished; **skips** experiments that already have `test_summary.json`.
- At the end of each experiment, `best.pth` (highest validation Dice) is loaded and evaluated on the test set.

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def evaluate(model, loader, full=False):
    model.eval()
    rows = []
    for b in loader:
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=CFG.USE_AMP and DEVICE.type == "cuda"):
            pred = model(b["image"].to(DEVICE)).argmax(1).cpu().numpy()
        for p, g, n in zip(pred, b["mask"].numpy(), b["name"]):
            if full:
                r = binary_metrics(p == 1, g == 1)
            else:
                s = (p == 1).sum() + (g == 1).sum()
                r = {"dsc": 1.0 if s == 0 else 2 * ((p == 1) & (g == 1)).sum() / s}
            r["name"] = n
            rows.append(r)
    model.train()
    return float(np.mean([r["dsc"] for r in rows])), rows


def run_test(model, out_dir):
    loader = DataLoader(BloodCellDataset(SPLIT["test"], build_eval_transform()), batch_size=8,
                        num_workers=CFG.NUM_WORKERS)
    _, rows = evaluate(model, loader, full=True)
    keys = ["dsc", "jaccard", "hd95", "asd"]
    pd.DataFrame(rows)[["name"] + keys].to_csv(out_dir / "test_per_image.csv", index=False)
    summary = {}
    for k in keys:
        v = np.array([r[k] for r in rows]) * (100 if k in ("dsc", "jaccard") else 1)  # DSC, Jaccard in %
        summary[k] = {"mean": float(v.mean()), "std": float(v.std())}
    (out_dir / "test_summary.json").write_text(json.dumps(summary, indent=2))
    return summary


def train_one(ratio, seed):
    out_dir = RUN_ROOT / f"{CFG.MODEL}_{ratio}p_s{seed}"
    if (out_dir / "test_summary.json").exists():
        print(f"== {out_dir.name}: already done, skipping ==")
        return json.loads((out_dir / "test_summary.json").read_text())
    out_dir.mkdir(parents=True, exist_ok=True)
    set_seed(seed)
    cfg_dump = {k: v for k, v in vars(CFG).items() if k.isupper()}
    cfg_dump.update({"AUG_" + k: v for k, v in vars(AUG).items() if k.isupper()})
    cfg_dump.update(ratio=ratio, seed=seed)
    (out_dir / "config.json").write_text(json.dumps(cfg_dump, indent=2, default=str))

    names = SPLIT["train"] if ratio == 100 else SPLIT[f"labeled_{ratio}"]
    if not names:
        raise ValueError(f"No labeled images at {ratio}% - check LABELED_RATIOS / the data.")
    print(f"\n== {out_dir.name}: {len(names)} labeled images, {CFG.MAX_ITERS} iterations ==")
    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(BloodCellDataset(names, build_train_transform()),
                              batch_size=min(CFG.BATCH_SIZE, len(names)), shuffle=True, drop_last=True,
                              num_workers=CFG.NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
                              generator=g, persistent_workers=CFG.NUM_WORKERS > 0)
    val_loader = DataLoader(BloodCellDataset(SPLIT["val"], build_eval_transform()), batch_size=8,
                            num_workers=CFG.NUM_WORKERS)

    model = build_model().to(DEVICE)
    opt = torch.optim.SGD(model.parameters(), lr=CFG.BASE_LR, momentum=CFG.MOMENTUM,
                          weight_decay=CFG.WEIGHT_DECAY)
    use_amp = CFG.USE_AMP and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=use_amp)

    start, best = 0, -1.0
    if (out_dir / "last.pth").exists():
        ck = torch.load(out_dir / "last.pth", map_location=DEVICE)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        start, best = ck["iter"], ck["best"]
        print(f"   resuming from iteration {start} (best val Dice {best:.4f})")
    log_f = open(out_dir / "log.csv", "a", newline="")
    log = csv.writer(log_f)
    if start == 0:
        log.writerow(["iter", "lr", "loss", "val_dice"])

    model.train()
    it_loader = infinite_loader(train_loader)
    running, t0 = 0.0, time.time()
    for it in range(start, CFG.MAX_ITERS):
        b = next(it_loader)
        img, msk = b["image"].to(DEVICE, non_blocking=True), b["mask"].to(DEVICE, non_blocking=True)
        lr = CFG.BASE_LR * (1 - it / CFG.MAX_ITERS) ** 0.9
        for pg in opt.param_groups:
            pg["lr"] = lr
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=use_amp):
            logits = model(img)
        loss = sup_loss(logits.float(), msk)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        running += loss.item()

        step = it + 1
        if step % CFG.LOG_INTERVAL == 0:
            print(f"   [{step:6d}/{CFG.MAX_ITERS}] loss {running / CFG.LOG_INTERVAL:.4f} | lr {lr:.5f} "
                  f"| {CFG.LOG_INTERVAL / (time.time() - t0):.1f} it/s")
            log.writerow([step, lr, running / CFG.LOG_INTERVAL, ""])
            running, t0 = 0.0, time.time()
        if step % CFG.VAL_INTERVAL == 0 or step == CFG.MAX_ITERS:
            vd, _ = evaluate(model, val_loader)
            log.writerow([step, "", "", vd]); log_f.flush()
            if vd > best:
                best = vd
                torch.save({"model": model.state_dict(), "iter": step, "val_dice": vd}, out_dir / "best.pth")
            print(f"   -> val Dice {vd:.4f} | best {best:.4f}")
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "iter": step, "best": best}, out_dir / "last.pth")
    log_f.close()

    model.load_state_dict(torch.load(out_dir / "best.pth", map_location=DEVICE)["model"])
    s = run_test(model, out_dir)
    print("   TEST:", "  ".join(f"{k.upper()} {v['mean']:.2f}({v['std']:.2f})" for k, v in s.items()))
    return s

## 8. Experiments at each label ratio

Rough estimate on a T4 GPU (not measured): UNeXt ~25-40 min / 30k iterations, Swin-Unet ~1.5-3 h / 30k iterations. If 12 h is not enough, split `LABELED_RATIOS` over several sessions.

In [ ]:
results = []
for ratio in CFG.LABELED_RATIOS:
    for seed in CFG.SEEDS:
        s = train_one(ratio, seed)
        results.append({"model": CFG.MODEL, "labeled_%": ratio, "seed": seed,
                        **{f"{k}_mean": v["mean"] for k, v in s.items()},
                        **{f"{k}_std": v["std"] for k, v in s.items()}})

## 9. Results

In [ ]:
# ---- Results table (paper format: mean(std)) ----
df = pd.DataFrame(results)
table = pd.DataFrame({
    "Model": df["model"], "Labeled %": df["labeled_%"], "Seed": df["seed"],
    **{k.upper(): [f"{m:.2f}({s:.2f})" for m, s in zip(df[f"{k}_mean"], df[f"{k}_std"])]
       for k in ["dsc", "jaccard", "hd95", "asd"]}})
table.to_csv(RUN_ROOT / f"summary_{CFG.MODEL}.csv", index=False)
display(table)

# ---- Training curves ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ratio in CFG.LABELED_RATIOS:
    log = pd.read_csv(RUN_ROOT / f"{CFG.MODEL}_{ratio}p_s{CFG.SEEDS[0]}" / "log.csv")
    tr, va = log.dropna(subset=["loss"]), log.dropna(subset=["val_dice"])
    axes[0].plot(tr["iter"], tr["loss"], label=f"{ratio}%")
    axes[1].plot(va["iter"], va["val_dice"], label=f"{ratio}%")
axes[0].set(title="Training loss", xlabel="Iteration"); axes[1].set(title="Validation Dice", xlabel="Iteration")
for ax in axes:
    ax.legend(title="Labeled"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(RUN_ROOT / f"curves_{CFG.MODEL}.png", dpi=150); plt.show()

In [ ]:
# ---- Test predictions: green = correct, red = false positive, blue = false negative ----
def show_predictions(ratio=None, n=4):
    ratio = ratio or CFG.LABELED_RATIOS[0]
    model = build_model(pretrained=False).to(DEVICE)
    ck = RUN_ROOT / f"{CFG.MODEL}_{ratio}p_s{CFG.SEEDS[0]}" / "best.pth"
    model.load_state_dict(torch.load(ck, map_location=DEVICE)["model"]); model.eval()
    ds = BloodCellDataset(SPLIT["test"][:n], build_eval_transform())
    fig, axes = plt.subplots(n, 3, figsize=(10, 3.3 * n))
    for i in range(n):
        b = ds[i]
        with torch.no_grad():
            p = model(b["image"][None].to(DEVICE)).argmax(1)[0].cpu().numpy().astype(bool)
        g = b["mask"].numpy().astype(bool)
        img = b["image"].permute(1, 2, 0).numpy() * np.array(AUG.STD) + np.array(AUG.MEAN)
        img = np.clip(img, 0, 1)
        err = np.zeros((*g.shape, 3)); err[p & g] = (0, 1, 0); err[p & ~g] = (1, 0, 0); err[~p & g] = (0, 0, 1)
        d = binary_metrics(p, g)["dsc"] * 100
        for ax, im, t in zip(axes[i], [img, g, err], ["Image", "Ground truth", f"Prediction (DSC {d:.1f}%)"]):
            ax.imshow(im, cmap="gray"); ax.set_title(t); ax.axis("off")
    plt.suptitle(f"{CFG.MODEL} – {ratio}% labeled"); plt.tight_layout(); plt.show()


show_predictions()

In [ ]:
# ---- Package results (without last.pth) -> download from the Output tab ----
zip_path = Path(CFG.WORK_DIR) / f"results_{CFG.MODEL}"
tmp = Path(CFG.WORK_DIR) / "_pack"
shutil.rmtree(tmp, ignore_errors=True)
shutil.copytree(RUN_ROOT, tmp, ignore=shutil.ignore_patterns("last.pth"))
shutil.make_archive(str(zip_path), "zip", tmp)
shutil.rmtree(tmp)
print("Created", zip_path.with_suffix(".zip"))